# M6S1B · Quantum computing with Quantum Inspire
**IE University · High-Performance Computing · 2026–27 · Guided practical (35–45 minutes)**

**Your mission:** Use a small quantum algorithm to identify whether a hidden function is **constant** or **balanced**. You will change **one line**, run the program twice, and explain the results.

We will use [Quantum Inspire](https://www.quantum-inspire.com/) from the Dutch QuTech/TU Delft ecosystem. It accepts **cQASM 3.0**, a short language for describing quantum circuits. You do **not** need Qiskit, Python packages, or access to the IE cluster for this exercise.

> **Important:** This notebook is a guided worksheet. The **quantum program runs in the Quantum Inspire website**, not inside Jupyter. Code boxes below are meant to be copied into its editor; do not try to execute them as Python cells.


## 1 · What you should learn (3 minutes)
By the end, you should be able to:

1. Explain that a **qubit** is a quantum information unit which can be in a superposition before measurement.
2. Recognise three gates: **X** (flips 0 and 1), **H** (changes the measurement basis and can create a superposition) and **CNOT** (conditionally flips another qubit).
3. Describe a real **quantum algorithm** and the answer it produces.
4. Explain why HPC and quantum processors may work together, and why this experiment does **not** prove a practical quantum speedup.

**The idea in plain language:** Imagine someone gives you a black box that takes one bit, 0 or 1, and outputs one bit. You want to know whether its answer is **always the same** or **changes with the input**.


## 2 · Meet the problem: two possible black boxes (4 minutes)
There are only two input values, so the possibilities below are easy to see.

| Black box | Output for input `0` | Output for input `1` | Type |
|:--|:--:|:--:|:--|
| A: `f(x) = 0` | 0 | 0 | **Constant**: same answer |
| B: `f(x) = x` | 0 | 1 | **Balanced**: one of each |

A classical program can always check both inputs, making **two calls** to the unknown function in the worst case. Deutsch's one-bit quantum algorithm can distinguish these two cases with **one oracle query**, assuming the correct quantum oracle is provided.

> **No magic:** The quantum computer does not print both function values at once. A carefully designed sequence of gates makes the **property we care about** appear in one measurement.

**Predict before continuing:** If you inspect only `f(0)` and receive `0`, do you know which box you have? Why not?


## 3 · Open Quantum Inspire (5 minutes)
1. Open **[Quantum Inspire](https://www.quantum-inspire.com/)** and select **My QI**.
2. Log in or create a **free account**. The platform does not offer anonymous execution.
3. Choose **New Project** and name it `HPC_M6S1B_Deutsch`.
4. Select **Create Algorithm** to open the **cQASM editor**.
5. When you run a program, choose the **QX emulator / simulator** (not physical hardware) and use **128 shots**. A shot means one repetition and measurement of the circuit.

If a button name or layout changes, follow the official [editor instructions](https://www.quantum-inspire.com/kbase/working-with-the-editor/) and [running instructions](https://www.quantum-inspire.com/kbase/executing-your-algorithm/). **Do not install anything or use access tokens.**

**If the platform is unavailable:** Read the code and finish the prediction questions. Record `NOT RUN — external service` rather than inventing experimental measurements.


## 4 · Warm-up: one quantum coin (optional, 5 minutes)
Copy this **complete** program into the Quantum Inspire editor. Select the QX emulator, use **128 shots**, and press **Run**.

```text
version 3.0
qubit q
bit b

H q
b = measure q
```

**Meaning of each line:**
- `version 3.0`: use the **current** Quantum Inspire language.
- `qubit q`: create one qubit (initially `|0⟩`).
- `bit b`: reserve one ordinary bit to store the result.
- `H q`: put the qubit in an equal superposition.
- `b = measure q`: obtain one ordinary `0` or `1`.

**Expected:** About half `0` and half `1` over many shots; **not** exactly 64 of each every time. The H gate creates a superposition, but each individual measurement still gives just one bit.

**Write down:** approximately what fraction of your shots gave `1`? ________


## 5 · Main experiment: Deutsch's algorithm (5 minutes to set up)
Create another algorithm (or replace the warm-up code). Copy **everything** in this block. The program starts with the **constant** black box; the optional CNOT line is commented out.

```text
version 3.0
qubit[2] q
bit[1] b

// Prepare the helper qubit as |1>
X q[1]

// Set up both qubits
H q[0]
H q[1]

// THE BLACK BOX (the oracle): f(x) = 0, constant
// For the balanced version, remove // on the next line:
// CNOT q[0], q[1]

// Finish the algorithm and read the answer
H q[0]
b[0] = measure q[0]
```

You need to remember just **one thing**: the **measured first qubit, `q[0]`, tells us which kind of function we have**. `0` means constant; `1` means balanced. The second qubit helps the algorithm work; we do not need to measure it.

> The program uses **Deutsch's algorithm**, the one-input-bit case of the more general Deutsch–Jozsa family. This is a teaching example, not a commercial HPC workload.


## 6 · Run A: the constant function (4 minutes)
1. Leave `// CNOT q[0], q[1]` **commented out**. The two slashes mean that line does nothing.
2. Press **Run**, select the **QX emulator**, choose **128 shots**, and start the job.
3. Open the completed **Job → Results** histogram. Look at the measured bit `b[0]`.
4. Record what you actually saw, not just what the worksheet predicts.

| Record A | Your observation |
|:--|:--|
| Oracle | Constant (`f(x) = 0`) |
| Expected result | `b[0] = 0` (essentially 100% on the ideal emulator) |
| Observed result and count/proportion | **Write here:** ______ |
| Job status | Completed / Failed / Not run: ______ |

**Why?** Without the CNOT, the first qubit goes through **H then H again**. The two H operations cancel, so it returns to `0`.


## 7 · Run B: the balanced function (4 minutes)
1. Find this line: `// CNOT q[0], q[1]`.
2. **Remove the two slashes** at the beginning. The line must now read **`CNOT q[0], q[1]`**. Do not change anything else.
3. Press **Run** again on the **QX emulator**, with **128 shots**.
4. Open the new completed **Job → Results** histogram and compare it with Run A.

| Record B | Your observation |
|:--|:--|
| Oracle | Balanced (`f(x) = x`) |
| Expected result | `b[0] = 1` (essentially 100% on the ideal emulator) |
| Observed result and count/proportion | **Write here:** ______ |
| Job status | Completed / Failed / Not run: ______ |

**Why did the answer change?** Because the helper qubit is prepared in a special state, the CNOT leaves a **phase difference** on the first qubit. The final H converts that phase information into a measurable `1`. You do **not** need to calculate the phase mathematically for this lesson.


## 8 · Understand the circuit in one table (3 minutes)
| Part of circuit | In simple terms | Why it matters |
|:--|:--|:--|
| `X q[1]` | Set the **helper** qubit to `1`. | Prepare the special input. |
| `H q[0]` and `H q[1]` | Prepare the qubits for the query. | Makes the oracle answer visible through interference. |
| **Oracle** (no gate or CNOT) | The **unknown function** we want to classify. | Exactly **one query** to that function. |
| Final `H q[0]` | Transform the hidden phase difference. | Turns the function's property into a measurable bit. |
| `measure q[0]` | Read the one-bit answer. | `0` = constant, `1` = balanced. |

### Two outcomes to remember
- **Constant oracle** → no CNOT → measured `0`.
- **Balanced oracle** → CNOT → measured `1`.

The key quantum effects are **superposition, phase kickback and interference**. Simply reading a superposition would not reveal both function outputs.


## 9 · Extra experiment (if you have time: 3 minutes)
Try a **different constant function**, `f(x) = 1`:

1. Return to the **constant** code (comment out the CNOT again).
2. Add `X q[1]` **inside the oracle section**, after the two H gates and before the final H.
3. Run again on the emulator.

The oracle now returns `1` for both inputs. **Prediction:** The measured answer is still `0`, because the function remains constant. The extra X changes an overall phase but not the classification.

**If you want a fourth case:** Leave the CNOT active **and** insert an additional X on `q[1]` in the oracle section. This implements `f(x) = 1 − x`, which is balanced; the measured answer should still be `1`.


## 10 · Link to your HPC course (3 minutes)
A real quantum computer is **not a replacement for an HPC cluster**. In a possible hybrid workflow:

1. **CPU/GPU:** Prepare scientific data, model and input parameters.
2. **Quantum processor or emulator:** Execute a carefully chosen quantum subroutine.
3. **CPU/GPU:** Collect many measurements and analyse the results.
4. **HPC:** Verify accuracy, compare against a classical method, and decide whether the extra overhead is worthwhile.

**Our exercise already resembles one part of this workflow:** We choose a program and job settings in a classical web interface, send the circuit to an emulator, and interpret a histogram of classical output bits.

**Critical caution:** This two-qubit demonstration **does not** establish faster end-to-end processing, lower energy use, or better results for practical HPC applications. Those questions require a fair comparison including preparation, communication, queueing and repeated measurement.

See the [EuroHPC quantum computing and access programme](https://www.eurohpc-ju.europa.eu/quantum-technologies/quantum-computing-access_en) for examples of quantum systems connected to supercomputers.


## 11 · Quick concept checks (5 minutes)
Choose one answer for each question **before** opening the solution. These are self-check questions, not a graded Blackboard attempt.

**Q1.** What is the purpose of the `H` gate in our circuit?  
A. It copies an unknown qubit.  B. It helps create/transform superposition and phase.  C. It permanently measures the qubit.
<details><summary>Reveal Q1 answer</summary>**B.** H is essential both before and after the oracle.</details>

**Q2.** What does one measurement of a qubit return?  
A. A normal bit, 0 or 1.  B. Every possible state at once.  C. An infinite-precision number.
<details><summary>Reveal Q2 answer</summary>**A.** You need many shots to estimate probabilities.</details>

**Q3.** In the main program, a `0` in `b[0]` means:  
A. The function is balanced.  B. The circuit failed.  C. The function is constant.
<details><summary>Reveal Q3 answer</summary>**C.** The final measurement classifies the function.</details>

**Q4.** What single change switches our oracle from `f(x)=0` to `f(x)=x`?  
A. Turn on the CNOT.  B. Delete the first H.  C. Measure twice.
<details><summary>Reveal Q4 answer</summary>**A.** Remove the `//` in front of `CNOT q[0], q[1]`.</details>

**Q5.** Why do we repeat a circuit many times (`shots`)?  
A. Because one measurement reads all amplitudes.  B. To observe an output distribution.  C. To train an AI model.
<details><summary>Reveal Q5 answer</summary>**B.** An ideal Deutsch classifier is deterministic, but shots help inspect the results and are essential in probabilistic circuits.</details>

**Q6.** Does a successful two-qubit demonstration prove quantum computers outperform HPC clusters?  
A. Yes, in every task.  B. Yes, because quantum gates have no overhead.  C. No, this is an educational oracle-query example.
<details><summary>Reveal Q6 answer</summary>**C.** Real advantage depends on workloads, errors, data movement and total costs.</details>


## 12 · Your short submission / exit ticket (3 minutes)
Write **four short sentences** in your own words:

1. In **Run A** I measured ______. This means the function was ______.
2. In **Run B** I measured ______. This means the function was ______.
3. The **one line** I changed was ______; it changed the oracle because ______.
4. This is related to HPC because ______, but it does **not** prove ______.

**Evidence (if requested by your professor):** one screenshot or the two observed histograms, plus the four sentences. Never share your account password or access token.

**If execution was blocked:** Report the error or `NOT RUN`, then fill in **predicted** results clearly labelled as predictions.


---
## Troubleshooting (read only if needed)
- **Syntax error?** Check that the first line is `version 3.0`, gate names use the correct capital letters, and the balanced line is exactly `CNOT q[0], q[1]`.
- **No run option?** Make sure you are logged in, have opened a project, and created an algorithm.
- **Hardware rejects a two-qubit program?** Choose the **QX emulator**. Physical backends have different qubit counts, connectivity and gate restrictions.
- **Job queued?** Wait for its status. A queue delay is not a measurement of quantum performance.
- **Unexpected result?** Recheck whether CNOT is commented out; also ensure you inspected **the first qubit**, `q[0]`, rather than the unmeasured helper.
- **Cannot access Quantum Inspire?** Complete the prediction and questions offline. Ask your instructor to show one live run in class.


## Sources and optional further practice
- [Quantum Inspire: Hello quantum world](https://www.quantum-inspire.com/kbase/hello-quantum-world/) — creating and measuring a two-qubit circuit.
- [Quantum Inspire: Deutsch–Jozsa algorithm](https://www.quantum-inspire.com/kbase/deutsch-jozsa-algorithm/) — source for the algorithm idea and constant/balanced black-box task.
- [Quantum Inspire: cQASM 3.0](https://www.quantum-inspire.com/kbase/cqasm/) — language guide.
- [Quantum Inspire: editor](https://www.quantum-inspire.com/kbase/working-with-the-editor/) and [running a job](https://www.quantum-inspire.com/kbase/executing-your-algorithm/) — current online workflow.
- [QuTech cQASM specification](https://qutech-delft.github.io/cQASM-spec/latest/) — optional reference for precise syntax.

**Attribution:** Educational exercise adapted from the public Quantum Inspire documentation of the Deutsch–Jozsa algorithm, with simplified code and original student instructions for M6S1B. All expected outcomes refer to an ideal noiseless emulator; actual external service execution was not performed by the notebook author.
